In [ ]:
import sys
from pathlib import Path

# Specify path to scripts dir for imports
scripts_dir = Path.cwd().parent / "scripts"
sys.path.insert(0, str(scripts_dir))

In [ ]:
import pandas as pd
import os

experiment_dir = "E:/Christine/100k_endometrium/"

# Path to cohort
df_path = os.path.join(experiment_dir, "endometrium_cohort_clean.csv")
patients_df = pd.read_csv(df_path)

In [ ]:
from plate_randomization import PlateConfig, PlatePlanner

cfg = PlateConfig(disease_tiles=5, healthy_tiles=5, seed=0)
mapping_path = "E:/Christine/100k_endometrium/ID0089_100K_endometrium.xlsx"

planner = PlatePlanner(patients_df, project_id="100k_DVP2_endometrium", project_dir=experiment_dir,
                       config=cfg, mapping_excel=mapping_path)

planner.summarize()

In [ ]:
# Get slides to pull, by box
plate = 4
plate_df = planner.get_rekvnr_on_plate(plate)     

In [ ]:
plate_df.head()

In [ ]:
# Prepare well assignment CSV
well_assignment = planner.prepare_well_assignment(plate)
well_assignment_path = planner.well_assignment_path(plate)

In [ ]:
well_assignment.head()

In [ ]:
from helper_functions import find_files_df

wsi_dir = r"J:\100k\100k_endometrium\slides\plate4"
# Path to WSI root dir
# wsi_dir = os.path.join(experiment_dir, "slides", f"plate{plate}")
suffix = ".mrxs"

slides_df, slide_paths = find_files_df(wsi_dir, suffix)
print(f"Number of {suffix} files found: ", len(slide_paths))

In [ ]:
import os

# Automatically generate paths
wsi_cache = os.path.join(experiment_dir, "cache_tissue_artifact_features.pkl")
zarr_dir = os.path.join(experiment_dir, "zarr")

In [ ]:
from tissue_artifact_segmentation import SegmentMany, get_failed_slides

# WSI preprocessing: tissue segmentation
segmenter = SegmentMany(slide_paths, wsi_cache, zarr_dir, "tissue", version="default", retry_on_previos_errors = True, force_rerun = False)

In [ ]:
from feature_extraction import ExtractMany

# Feature extraction
foundation_model = 'h-optimus-0'

extractor = ExtractMany(wsi_paths = slide_paths, cache_path = wsi_cache, zarr_dir = zarr_dir, foundation_model = foundation_model, mpp=0.5, tile_px = 224, remove_artifacts = False, retry_on_previous_errors = True, use_cache=True, force_rerun = False)

In [ ]:
failed = get_failed_slides(wsi_cache, slide_paths)
failed["slide_path"].unique().tolist()

In [ ]:
path = slide_paths[15]
print(path)

In [ ]:
from wsidata import open_wsi
import lazyslide as zs

path = failed["slide_path"][0]
zarr_path = os.path.join(zarr_dir, os.path.basename(path).replace(".mrxs", ".zarr"))
wsi = open_wsi(path, zarr_path, attach_images=True)
wsi

In [ ]:
zs.pp.find_tissues(wsi, method="entropy", level=-5, key_added="tissue_entropy")

In [ ]:
zs.pl.tissue(wsi, tissue_id = 0, tissue_key = 'tissue_entropy', show_contours=True)

In [ ]:
wsi.write(zarr_path)

In [ ]:
from tissue_artifact_segmentation import SegmentMany

# WSI preprocessing: artifact segmentation
segmenter = SegmentMany(slide_paths, wsi_cache, zarr_dir, "artifact", version="default", retry_on_previos_errors = True)

In [ ]:
from abmil import ABMILInference

# Run inference with trained ABMIL
abmil_path = "E:/Christine/abmil_hopt_mpp05_binary.pt"
inference_cache = os.path.join(experiment_dir, "cache_abmil_inference.pkl")
heatmap_dir = os.path.join(experiment_dir, "heatmaps")

In [ ]:
inference = ABMILInference(checkpoint_path=abmil_path, zarr_dir=zarr_dir, slides=slide_paths, cache_path=inference_cache, save_heatmap=True, heatmap_dir = heatmap_dir)
inference.process_slides()

In [ ]:
plate_dir = os.path.join(experiment_dir, f"P{plate}")

In [ ]:
# Save inference results
inference_output = os.path.join(plate_dir, f"inference_result_P{plate}.csv")
skipped_output = os.path.join(plate_dir, f"inference_skipped_slides_P{plate}.csv")

df_inference = inference.results_dataframe()
df_inference.to_csv(inference_output, index=False)

skipped_slides = inference.get_skipped_slides()
skipped_slides.to_csv(skipped_output, index=False)
print(f"Plate {plate}, Skipped slides: {len(skipped_slides)}")

In [ ]:
well_assignment_path = r"E:\Christine\100k_endometrium\P4\100k_DVP2_endometrium_well_assignment_P4.csv"

In [ ]:
from plate_randomization import load_well_assignment

wa = load_well_assignment(well_assignment_path)
print(wa.head())

In [ ]:
# Change slide idx
slide = slide_paths[4]
print(slide)

In [ ]:
from roi_selection import ROISelector

selector = ROISelector(inference_cache, slide, disease_k=cfg.disease_tiles, healthy_k=cfg.healthy_tiles, random_state = 42, attention_floor = 0.05, strict_topk = False)

In [ ]:
selector.tiles_to_cut(save=False)

In [ ]:
viewer, url = selector.launch_calibration_viewer(plate_dir)

In [ ]:
# ...calibrate in the browser, Save & close...
# manually add calibration points (3 points, easy to locate on LMD)
selector.write_lmd_and_metadata(well_assignment_path = well_assignment_path, overwrite = False)

In [ ]:
well_assignment_path = r"E:\Christine\100k_endometrium\P3\100k_DVP2_endometrium_well_assignment_P3.csv"

In [ ]:
from plate_randomization import load_well_assignment
from evosep_import import make_evosep_import, write_evosep_csv

prefix = "100k_endometrium_AST4_ENO0_120SPD"
wa = load_well_assignment(well_assignment_path)
evo, wa_ms = make_evosep_import(wa, date="20261120", seed=1, prefix=prefix, slots={"Q1": 3, "Q2": 4, "Q3": 1, "Q4": 2})
write_evosep_csv(evo, r"E:/Christine/100k_endometrium/P3/evosep_import_100k_E3.csv")

In [ ]:
set(wa_ms["ms_sample_name"]) == set(evo.loc[~evo["Sample Name"].str.contains("_QC_"), "Sample Name"])

In [ ]:
print(evo.head())